In [1]:
import numpy as np, pandas as pd, torch
from deepfake.data import frequency_ordered_nodes
from deepfake.constants import WAVELET_LEVEL

In [2]:
nodes = np.array(frequency_ordered_nodes(WAVELET_LEVEL))
run_dir = "results/train_entropy_based/runs/db4/seed_42"

In [3]:
# nós escolhidos no treino (acumulado até a melhor época)
state = torch.load(f"{run_dir}/model.pt", weights_only=True)
train_counts = pd.Series(state["features.2.subband_selection"].numpy(), index=nodes)

In [ ]:
# nós e scores de cada áudio no eval
meta = pd.read_csv("datasets/waveforms/eval_2019_meta.csv")

meta["node"] = nodes[np.load(f"{run_dir}/selection_eval_2019.npy")]
meta["score"] = np.load(f"{run_dir}/scores_eval_2019.npy")

meta["node"].value_counts(normalize=True)                                      # distribuição geral

node
daa    0.754060
dad    0.122858
dda    0.034364
ddd    0.031809
ada    0.026335
add    0.019793
aad    0.010753
aaa    0.000028
Name: proportion, dtype: float64

In [5]:
pd.crosstab(meta["is_bonafide"], meta["node"], normalize="index")              # bonafide × spoof

node,aaa,aad,ada,add,daa,dad,dda,ddd
is_bonafide,,,,,,,,
False,0.000016,0.010347,0.025156,0.019364,0.757162,0.124057,0.032670,0.031229
True,0.000136,0.014276,0.036574,0.023521,0.727124,0.112441,0.049082,0.036846


In [6]:
pd.crosstab(meta["attack"], meta["node"], normalize="index")[nodes]            # ataque × nó (heatmap) 

node,aaa,aad,add,ada,dda,ddd,dad,daa
attack,,,,,,,,
-,0.000136,0.014276,0.023521,0.036574,0.049082,0.036846,0.112441,0.727124
A07,0.000000,0.006105,0.014042,0.025438,0.038462,0.018519,0.129833,0.767603
A08,0.000000,0.007937,0.049044,0.037851,0.040293,0.039276,0.108262,0.717338
A09,0.000000,0.006919,0.016280,0.018519,0.018519,0.042125,0.028287,0.869353
A10,0.000000,0.007937,0.015263,0.025438,0.040090,0.035613,0.120472,0.755189
A11,0.000000,0.008954,0.008140,0.018315,0.036020,0.031136,0.086895,0.810541
A12,0.000000,0.007937,0.017298,0.031136,0.040090,0.042125,0.110297,0.751119
A13,0.000000,0.001628,0.002035,0.009361,0.025031,0.058812,0.239316,0.663818
A14,0.000000,0.005698,0.011600,0.016280,0.007530,0.009361,0.056777,0.892755


In [ ]:
def validade_node_stats(node, is_correct, data):
    node = f"{node}_{"correct" if is_correct else "wrong"}"
    if not node in data:
        data[node] = 0
    data[node] += 1
    return data

def model_stats():
    stats = {
        "bonafide_correct": 0,
        "bonafide_wrong": 0,
        "spoof_correct": 0,
        "spoof_wrong": 0,
    }
    nodes_stats = {}

    for line in meta.iterrows():
        y_hat = line[1]["score"] > 0

        if line[1]["is_bonafide"]:
            if y_hat:
                stats["bonafide_correct"] += 1
                nodes_stats = validade_node_stats(line[1]["node"], True, nodes_stats)
            else:
                stats["bonafide_wrong"] += 1
                nodes_stats = validade_node_stats(line[1]["node"], False, nodes_stats)
        else:
            if y_hat:
                stats["spoof_wrong"] += 1
                nodes_stats = validade_node_stats(line[1]["node"], False, nodes_stats)
            else:
                stats["spoof_correct"] += 1
                nodes_stats = validade_node_stats(line[1]["node"], True, nodes_stats)
    print(stats)
    print(nodes_stats)

    nodes_porcentage = {
        node: float(value / meta["node"].value_counts()[node.split("_")[0]])
        for node, value in nodes_stats.items()
    }
    print(nodes_porcentage)


model_stats()

{'bonefide_correct': 5772, 'bonafide_wrong': 1583, 'spoof_correct': 48795, 'spoof_wrong': 15087}
{'daa_correct': 39065, 'dad_correct': 7915, 'daa_wrong': 14652, 'ada_correct': 1639, 'dad_wrong': 837, 'dda_correct': 2121, 'ddd_correct': 2012, 'dda_wrong': 327, 'add_correct': 1199, 'aad_correct': 615, 'ada_wrong': 237, 'ddd_wrong': 254, 'aad_wrong': 151, 'add_wrong': 211, 'aaa_correct': 1, 'aaa_wrong': 1}
{'daa_correct': 0.7272371874825474, 'dad_correct': 0.9043647166361974, 'daa_wrong': 0.2727628125174526, 'ada_correct': 0.8736673773987207, 'dad_wrong': 0.09563528336380256, 'dda_correct': 0.866421568627451, 'ddd_correct': 0.8879082082965578, 'dda_wrong': 0.13357843137254902, 'add_correct': 0.850354609929078, 'aad_correct': 0.8028720626631853, 'ada_wrong': 0.1263326226012793, 'ddd_wrong': 0.11209179170344219, 'aad_wrong': 0.19712793733681463, 'add_wrong': 0.14964539007092198, 'aaa_correct': 0.5, 'aaa_wrong': 0.5}


In [ ]:
["aaa"]

np.int64(2)